<a href="https://colab.research.google.com/github/jakubs692/University/blob/main/DNN/bootcamp-ml/Lab_4_softmax_regression_student_version.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><img src='https://drive.google.com/uc?id=1_utx_ZGclmCwNttSe40kYA6VHzNocdET' height="60"></center>

AI TECH - Akademia Innowacyjnych Zastosowań Technologii Cyfrowych. Programu Operacyjnego Polska Cyfrowa na lata 2014-2020
<hr>

<center><img src='https://drive.google.com/uc?id=1BXZ0u3562N_MqCLcekI-Ens77Kk4LpPm'></center>

<center>
Projekt współfinansowany ze środków Unii Europejskiej w ramach Europejskiego Funduszu Rozwoju Regionalnego
Program Operacyjny Polska Cyfrowa na lata 2014-2020,
Oś Priorytetowa nr 3 "Cyfrowe kompetencje społeczeństwa" Działanie  nr 3.2 "Innowacyjne rozwiązania na rzecz aktywizacji cyfrowej"
Tytuł projektu:  „Akademia Innowacyjnych Zastosowań Technologii Cyfrowych (AI Tech)”
    </center>

# Softmax regression

In this exercise you will train a softmax regression model to recognize handwritten digits.
  
The general setup is as follows:
* we are given a set of pairs $(x, y)$, where $x \in R^D$ is a vector of real numbers representing the features, and $y \in \{1,...,c\}$ is the target (in our case we have ten classes, so $c=10$),
* for a given $x$ we model the probability of $y=j$ by $$h(x)_j=p_j = \frac{e^{w_j^Tx}}{\sum_{i=1}^c e^{w_i^Tx}},$$
* to find the right $w$ we will optimize the so called multiclass log loss:
$$L(y,p) = \log{p_y},$$
$$J(w) = -\frac{1}{n}\sum_{i=1}^n L(y_i,h(x)),$$
* with the loss function in hand we can improve our guesses iteratively:
    * $w_{ij}^{t+1} = w_{ij}^t - \text{step_size} \cdot \frac{\partial J(w)}{\partial w_{ij}}$,
* we can end the process after some predefined number of epochs (or when the changes are no longer meaningful).

Let's start with importing the MNIST dataset.

In [103]:
!wget -O mnist.npz https://s3.amazonaws.com/img-datasets/mnist.npz
!pip install plotly

--2026-10-31 21:43:39--  https://s3.amazonaws.com/img-datasets/mnist.npz
Resolving s3.amazonaws.com (s3.amazonaws.com)... 16.15.245.213, 16.182.70.224, 16.15.255.38, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|16.15.245.213|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11490434 (11M) [application/octet-stream]
Saving to: ‘mnist.npz’

mnist.npz           100%[===================>]  10.96M  41.2MB/s    in 0.3s    

2026-10-31 21:43:40 (41.2 MB/s) - ‘mnist.npz’ saved [11490434/11490434]



In [104]:
import numpy as np

def load_mnist(path='mnist.npz'):
    with np.load(path) as f:
        x_train, _y_train = f['x_train'], f['y_train']
        x_test, _y_test = f['x_test'], f['y_test']

    x_train = x_train.reshape(-1, 28 * 28) / 255.
    x_test = x_test.reshape(-1, 28 * 28) / 255.


    y_train = np.zeros((_y_train.shape[0], 10))
    y_train[np.arange(_y_train.shape[0]), _y_train] = 1

    y_test = np.zeros((_y_test.shape[0], 10))
    y_test[np.arange(_y_test.shape[0]), _y_test] = 1

    return (x_train, y_train), (x_test, y_test)

(x_train, y_train), (x_test, y_test) = load_mnist()

Let's take a look at the data. In the "x" arrays you'll find the images (encoded as pixel intensities) and in the "y" ones you'll find the labels (one-hot encoded).

In [105]:
print(x_train.shape)
print(y_train.shape)

print(x_train[:10])
print(y_train[:10])

(60000, 784)
(60000, 10)
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
[[0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]]


Now let us see the data in a more human way.

In [106]:
import numpy as np
if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_

from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

num_samples = 10
plots = make_subplots(rows=1, cols=num_samples)

for i in range(num_samples):
  a = x_train[i, :].reshape(28,28)
  img = go.Heatmap(z=a, colorscale='gray')
  plots.add_trace(img, row=1, col=i+1)

plots.update_yaxes(autorange='reversed', scaleanchor='x', constrain='domain')
plots.update_xaxes(constrain='domain')
plots.update_traces(showscale=False)
plots.show()

In [107]:
x_train = np.hstack([np.ones((x_train.shape[0], 1)), x_train])
x_test = np.hstack([np.ones((x_test.shape[0], 1)), x_test])

print(x_train.shape, x_test.shape)


(60000, 785) (10000, 785)


Next, we prepare $X$ and $y$ variables

In [108]:
X = x_train[:4000]
y = y_train[:4000]

print(X.shape)
print(y.shape)

(4000, 785)
(4000, 10)


To train the model we will (obviously) use gradient descent. Inside the loop we need a method to compute the gradients. Let's start with implementing it, together with some helper functions.

In [109]:
# We will store the weights in a D x c matrix, where D is the number of features, and c is the number of classes
#weights = (...) # TODO: Fill in, be sure to have the right shape!

D = X.shape[1]
c = y.shape[1]
weights = np.zeros([D, c]) # (D,c)

assert weights.shape == (D,c)

def softmax(z):
    ########################################
    # TODO: implement the softmax function #
    ########################################
    z = z - z.max(1, keepdims=True)
    return np.exp(z) / np.sum(np.exp(z), axis = 1, keepdims=True)  # (N,c)


def predict(weights, X):
    ###################################
    # TODO: compute the probabilities #
    ###################################
    return softmax(X@weights)          # softmax((N,D)@(D,c)) (N,c)

def compute_loss_and_gradients(weights, X, y, l2_reg):
    #############################################################################
    # TODO: compute loss and gradients, don't forget to include regularization! #
    #############################################################################
    N = X.shape[0]
    p = predict(weights, X) # (N,c)
    loss = -np.sum(y * np.log(p)) / N + 0.5 * l2_reg * np.sum(weights[1:]**2)

    reg_grad = l2_reg * weights
    reg_grad[0] = 0                      # bias row: no regularization
    grad = X.T@(p - y) / N + reg_grad     # (D,c)

    return (loss, grad)


**my note**  z = z - z.max(1, keepdims=True) is not necessary but improves numerical stability

We are now in position to complete the training pipeline.

If you have problems with convergence, be sure to check the gradients numerically.

In [110]:
l2_reg = 0.5
n_epochs = 250
lr = 0.05
t = 0.99

losses = []
for i in range(n_epochs):
    loss, grad = compute_loss_and_gradients(weights, X, y, l2_reg)
    losses.append(loss)

    weights -= lr * grad
    lr *= t

fig = px.line(x=range(1,n_epochs+1), y=losses)
layout = go.Layout(xaxis_title="Epoch", yaxis_title='Loss')
fig.update_layout(layout)

fig.show()

Now compute your accuracy on the training and test sets.

In [111]:
acc = np.mean(predict(weights, x_train).argmax(axis=1) == y_train.argmax(axis=1))
print("Train accuracy: ", acc)
acc = np.mean(predict(weights, x_test).argmax(axis=1) == y_test.argmax(axis=1))
print("Test accuracy: ", acc)

Train accuracy:  0.8017333333333333
Test accuracy:  0.807


We can also visualize the weights learned by our algorithm. Try to anticipate the result before executing the cell below.

In [112]:
num_samples = 10
plots = make_subplots(rows=1, cols=num_samples)

for i in range(num_samples):
  a = weights[1:, i].reshape(28,28)
  img = go.Heatmap(z=a, colorscale='gray')
  plots.add_trace(img, row=1, col=i+1)

plots.update_yaxes(autorange='reversed', scaleanchor='x', constrain='domain')
plots.update_xaxes(constrain='domain')
plots.update_traces(showscale=False)
plots.show()

Note that we only used a small portion of the data to develop the model. Now, implement the training on full data. Also, validate your model properly and find a good value for `l2_reg` hyperparameter. Try to experiment with `batch_size`.


In [113]:

D = x_train.shape[1]
c = y.shape[1]
weights = np.zeros([D, c]) # (D,c)

In [114]:
################################################
# TODO: implement the proper training pipeline #
################################################

def full_train(weights, X_matrix, y_matrix, batch_size, lr, l2_reg, n_epochs, t=0.99, X_validation=None, y_validation=None):

    weights = weights.copy()
    N = X_matrix.shape[0]
    train_losses, val_accs = [], []

    for epoch in range(n_epochs):
        perm = np.random.permutation(N)
        epoch_loss = []
        for i in range(0, N, batch_size):
            batch = perm[i:i+batch_size]
            loss, grad = compute_loss_and_gradients(weights, X_matrix[batch], y_matrix[batch], l2_reg)
            weights -= lr * grad
            epoch_loss.append(loss)
        lr *= t
        train_losses.append(np.mean(epoch_loss))
        if X_validation is not None:
            val_accs.append(np.mean(predict(weights, X_validation).argmax(1) == y_validation.argmax(1)))

    return weights, train_losses, val_accs



In [115]:
np.random.seed(0)
perm = np.random.permutation(x_train.shape[0])
n_val = int(0.2 * x_train.shape[0])
val_idx, tr_idx = perm[:n_val], perm[n_val:]
X_tr, y_tr = x_train[tr_idx], y_train[tr_idx]
X_val, y_val = x_train[val_idx], y_train[val_idx]

lr = 0.5
batch_size = 50
n_epochs = 200
t = 0.99

# sweep l2
results = {}
for l2 in [0, 1e-5, 1e-4, 1e-3, 1e-2]:
    w, losses, val_accs = full_train(np.zeros((D, c)), X_tr, y_tr, batch_size, lr, l2, n_epochs, t=t,
                                     X_validation=X_val, y_validation=y_val)
    val_loss, _ = compute_loss_and_gradients(w, X_val, y_val, 0)
    results[l2] = (w, losses, val_accs, val_loss)
    print(f"l2={l2:<7} train loss={losses[-1]:.4f}  val loss={val_loss:.4f}  val acc={val_accs[-1]:.4f}")

best_l2 = min(results, key=lambda k: results[k][3])
print("best l2:", best_l2)

# validation accuracy per epoch
fig = go.Figure()
for l2, (_, _, val_accs, _) in results.items():
    fig.add_scatter(x=list(range(1, n_epochs + 1)), y=val_accs, name=f"l2={l2}")
fig.update_layout(xaxis_title="Epoch", yaxis_title="Validation accuracy")
fig.show()

# retraining on all training data with the chosen l2
final_w, final_losses, _ = full_train(np.zeros((D, c)), x_train, y_train, batch_size, lr, best_l2, n_epochs, t=t)
test_acc = np.mean(predict(final_w, x_test).argmax(1) == y_test.argmax(1))
print(f"Test accuracy (l2={best_l2}): {test_acc:.4f}")


l2=0       train loss=0.2184  val loss=0.2859  val acc=0.9223
l2=1e-05   train loss=0.2283  val loss=0.2777  val acc=0.9247
l2=0.0001  train loss=0.2629  val loss=0.2656  val acc=0.9266
l2=0.001   train loss=0.3416  val loss=0.2836  val acc=0.9214
l2=0.01    train loss=0.5523  val loss=0.3934  val acc=0.9011
best l2: 0.0001


Test accuracy (l2=0.0001): 0.9267


<center><img src='https://drive.google.com/uc?id=1BXZ0u3562N_MqCLcekI-Ens77Kk4LpPm'></center>